**Week 7 — EDA, Bootstrap CI, t-test และ Chi-square (ข้อมูลลูกค้า/แคมเปญ)**

### สรุปเงื่อนไขการประมวลผลข้อมูล
- **การอ่านไฟล์:**
  - ข้อมูลลูกค้า: `cust = pd.read_csv('biz_customers.csv')` (1203 แถว แถวละ 1 คน)
  - ข้อมูลแคมเปญ: `camp = pd.read_csv('biz_campaigns.csv')` (240 แถว แถวละ 1 แคมเปญ)
  - ข้อมูลผลสุ่ม Bootstrap: `boot_df = pd.read_csv('pb07_boot_diff.csv')` (1000 รอบ ใช้ค่าในไฟล์โดยตรง ไม่ต้องสุ่มใหม่)
- **การนับดัชนี (0-based Indexing):** เริ่มนับจาก 0 ทั้งแถวและคอลัมน์
- **การเลือกช่วงแถว:** ข้อที่ระบุ "แถวที่ a ถึง b" ให้ใช้ `.iloc[a:b+1]` (รวมทั้งแถว a และแถว b)
- **การทดสอบสมมติฐาน:** ใช้ฟังก์ชันจาก `scipy.stats` (เช่น `ttest_ind`, `ttest_1samp`, `chi2_contingency`) ด้วยค่าเริ่มต้นตามที่โจทย์ระบุ

### รายละเอียดไฟล์ข้อมูล

**1. `biz_customers.csv`** (1203 แถว)
- `age`: อายุของลูกค้า (ปี)
- `income_k`: รายได้ต่อเดือน (พันบาท)
- `tenure_months`: ระยะเวลาที่เป็นลูกค้า (เดือน)
- `channel`: ช่องทางที่ซื้อบ่อย (`online` = ออนไลน์, `store` = หน้าร้าน)
- `loyalty_member`: สมาชิกสะสมแต้ม (1 = เป็น, 0 = ไม่เป็น)
- `web_visits`: จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด
- `email_opens`: จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ)
- `basket_size`: จำนวนสินค้าเฉลี่ยต่อออเดอร์ (ขนาดตะกร้า)
- `spend_k`: ยอดซื้อต่อเดือน (พันบาท)
- `orders_90d`: จำนวนออเดอร์ใน 90 วันล่าสุด
- `churn`: ลูกค้าเลิกใช้บริการหรือไม่ (1 = เลิก, 0 = ไม่เลิก)
- `responded`: ลูกค้าตอบรับแคมเปญโปรโมชันหรือไม่ (1 = ตอบรับ, 0 = ไม่ตอบรับ)

**2. `biz_campaigns.csv`** (240 แถว)
- `season`: ไตรมาสของแคมเปญ (`Q1`–`Q4`)
- `sales`: ยอดขายของแคมเปญ (พันชิ้น)

**3. `pb07_boot_diff.csv`** (1000 แถว)
- `boot_diff`: ผลต่างของ basket_size เฉลี่ยระหว่างสมาชิก (loyalty=1) ลบ ไม่ใช่สมาชิก (loyalty=0) จากการทำ Bootstrap $B = 1000$ รอบ

**โจทย์ข้อ 7.1:** ฝึกใช้เฉพาะลูกค้าที่เป็นสมาชิกสะสมแต้ม (`loyalty_member == 1`) ในตาราง cust หาค่าสหสัมพันธ์เพียร์สัน (Pearson correlation) ระหว่าง `email_opens` กับ `web_visits` = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านข้อมูลลูกค้าและกรองเฉพาะ loyalty_member == 1
cust = pd.read_csv('biz_customers.csv')
cust_loyalty = cust[cust['loyalty_member'] == 1]
print(f"จำนวนสมาชิกสะสมแต้ม: {len(cust_loyalty)} คน")

# 2. คำนวณ Pearson correlation ระหว่าง email_opens กับ web_visits
corr_matrix = cust_loyalty[['email_opens', 'web_visits']].corr()
print("Correlation matrix:")
print(corr_matrix)

ans_71 = corr_matrix.loc['email_opens', 'web_visits']
print(f"คำตอบข้อ 7.1: {ans_71:.4f}")


จำนวนสมาชิกสะสมแต้ม: 453 คน
Correlation matrix:
             email_opens  web_visits
email_opens     1.000000    0.136894
web_visits      0.136894    1.000000
คำตอบข้อ 7.1: 0.1369


**โจทย์ข้อ 7.2:** ฝึกหาค่าความเบ้ (skewness) ของ `spend_k` ของลูกค้าทุกคนในตาราง cust ด้วย `.skew()` ของ pandas (ค่าเริ่มต้น) = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านข้อมูลลูกค้าทั้งหมด
cust = pd.read_csv('biz_customers.csv')

# 2. คำนวณความเบ้ (Skewness) ของ spend_k ด้วย .skew()
skew_spend = cust['spend_k'].skew()
print(f"ค่าความเบ้ของ spend_k: {skew_spend:.4f}")
print(f"คำตอบข้อ 7.2: {skew_spend:.4f}")


ค่าความเบ้ของ spend_k: 0.3807
คำตอบข้อ 7.2: 0.3807


**โจทย์ข้อ 7.3:** ฝึกใช้ไฟล์ `biz_customers.csv` ทำ two-sample t-test (`stats.ttest_ind` ด้วยค่าเริ่มต้น) เปรียบเทียบ `orders_90d` ของกลุ่ม `responded = 1` (ใส่เป็นกลุ่มแรก) กับ `responded = 0` ของลูกค้าทุกคนในไฟล์ ค่า t-statistic = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. แบ่งข้อมูลตามกลุ่ม responded
cust = pd.read_csv('biz_customers.csv')
g1 = cust[cust['responded'] == 1]['orders_90d']
g0 = cust[cust['responded'] == 0]['orders_90d']
print(f"จำนวน responded=1: {len(g1)}, responded=0: {len(g0)}")

# 2. ทำ Two-sample t-test (ใส่ responded=1 เป็นกลุ่มแรก)
ttest_res = stats.ttest_ind(g1, g0)
print(f"t-statistic: {ttest_res.statistic:.4f}")
print(f"p-value: {ttest_res.pvalue:.4e}")
print(f"คำตอบข้อ 7.3: {ttest_res.statistic:.4f}")


จำนวน responded=1: 371, responded=0: 832
t-statistic: 2.8220
p-value: 4.8505e-03
คำตอบข้อ 7.3: 2.8220


**โจทย์ข้อ 7.4:** กรณีศึกษาทีมการตลาดของเครือร้านค้าปลีกสงสัยว่าจำนวนครั้งที่เข้าเว็บ/แอปใน 30 วัน `web_visits` โดยเฉลี่ยของลูกค้าที่ตอบรับโปรโมชัน (`responded == 1`) "สูงกว่า" ลูกค้าที่ไม่ตอบรับโปรโมชัน (`responded == 0`) จึงดูเฉพาะ**ลูกค้าแถวที่ 741 ถึง 1040** ของตาราง cust แล้วทดสอบ $H_0: \mu_1 = \mu_2$ เทียบกับ $H_1: \mu_1 > \mu_2$ ด้วย `stats.ttest_ind(..., alternative='greater')` โดยใส่ลูกค้าที่ตอบรับโปรโมชัน (`responded == 1`) เป็นกลุ่มแรก (พารามิเตอร์อื่นเป็นค่าเริ่มต้น) ค่า p-value = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. เลือกแถวที่ 741 ถึง 1040 (ดัชนี 0-based รวมปลาย ใช้ .iloc[741:1041])
cust = pd.read_csv('biz_customers.csv')
sub_cust = cust.iloc[741:1041]
print(f"ขนาดชุดข้อมูลย่อย: {sub_cust.shape}")

# 2. แยกกลุ่ม responded == 1 และ responded == 0
g1 = sub_cust[sub_cust['responded'] == 1]['web_visits']
g0 = sub_cust[sub_cust['responded'] == 0]['web_visits']
print(f"จำนวน responded=1: {len(g1)}, responded=0: {len(g0)}")

# 3. ทดสอบแบบด้านเดียว (alternative='greater')
res_74 = stats.ttest_ind(g1, g0, alternative='greater')
print(f"t-statistic: {res_74.statistic:.4f}")
print(f"p-value: {res_74.pvalue:.4f}")
print(f"คำตอบข้อ 7.4: {res_74.pvalue:.4f}")


ขนาดชุดข้อมูลย่อย: (300, 13)
จำนวน responded=1: 89, responded=0: 211
t-statistic: 2.5299
p-value: 0.0060
คำตอบข้อ 7.4: 0.0060


**โจทย์ข้อ 7.5:** กรณีศึกษาฝ่ายวางแผนการผลิตของบริษัทเครื่องดื่มตั้งเป้าว่าแคมเปญที่ทำในไตรมาส Q2 (`season == 'Q2'`) ควรมียอดขาย (`sales`, พันชิ้น) เฉลี่ยแคมเปญละ 18 พันชิ้น ทีมวิเคราะห์จึงทดสอบ $H_0: \mu = 18$ เทียบกับ $H_1: \mu \ne 18$ (สองด้าน) ด้วย `stats.ttest_1samp` กับแคมเปญไตรมาส Q2 ทุกแคมเปญในตาราง camp ค่า p-value = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. อ่านข้อมูลแคมเปญและกรองเฉพาะ season == 'Q2'
camp = pd.read_csv('biz_campaigns.csv')
camp_q2 = camp[camp['season'] == 'Q2']
print(f"จำนวนแคมเปญในไตรมาส Q2: {len(camp_q2)}")

# 2. One-sample t-test เทียบกับ mu = 18 (สองด้าน)
res_75 = stats.ttest_1samp(camp_q2['sales'], popmean=18)
print(f"ยอดขายเฉลี่ยใน Q2: {camp_q2['sales'].mean():.4f}")
print(f"t-statistic: {res_75.statistic:.4f}")
print(f"p-value: {res_75.pvalue:.4f}")
print(f"คำตอบข้อ 7.5: {res_75.pvalue:.4f}")


จำนวนแคมเปญในไตรมาส Q2: 45
ยอดขายเฉลี่ยใน Q2: 19.3778
t-statistic: 1.7995
p-value: 0.0788
คำตอบข้อ 7.5: 0.0788


**โจทย์ข้อ 7.6:** กรณีศึกษาฝ่ายวิเคราะห์ลูกค้าของแพลตฟอร์มอีคอมเมิร์ซสุ่มลูกค้า 120 คน แล้วต้องการประเมินความไม่แน่นอนของผลต่างของจำนวนสินค้าเฉลี่ยต่อออเดอร์ `basket_size` เฉลี่ย ระหว่างลูกค้าที่เป็นสมาชิกสะสมแต้ม (`loyalty_member == 1`) ลบลูกค้าที่ไม่ใช่สมาชิก (`loyalty_member == 0`) (ค่าเฉลี่ยกลุ่มแรก − ค่าเฉลี่ยกลุ่มหลัง) จึงทำ bootstrap (สุ่มลูกค้าแบบใส่คืน ขนาด 120) จำนวน $B = 1000$ รอบ เก็บผลต่างของแต่ละรอบไว้ในไฟล์ `pb07_boot_diff.csv` (คอลัมน์ `boot_diff`) สร้าง **90% Bootstrap CI** ด้วย percentile method (`np.percentile` ค่าเริ่มต้น) **ขอบบนของช่วง** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ผลต่าง bootstrap 1000 รอบ
boot_df = pd.read_csv('pb07_boot_diff.csv')
diffs = boot_df['boot_diff']
print(f"จำนวนรอบ bootstrap: {len(diffs)}")

# 2. คำนวณช่วงความเชื่อมั่น 90% (Percentile Method)
# ระดับความเชื่อมั่น 90% -> alpha = 10% -> ตัดหัวท้ายข้างละ 5%
ci_lower = np.percentile(diffs, 5)
ci_upper = np.percentile(diffs, 95)
print(f"90% Bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"ขอบบนของช่วง (95th percentile): {ci_upper:.4f}")
print(f"คำตอบข้อ 7.6: {ci_upper:.4f}")


จำนวนรอบ bootstrap: 1000
90% Bootstrap CI: [-0.2634, 0.2768]
ขอบบนของช่วง (95th percentile): 0.2768
คำตอบข้อ 7.6: 0.2768


**โจทย์ข้อ 7.7:** กรณีศึกษาผู้จัดการร้านค้าปลีกอยากรู้ว่าช่องทางที่ซื้อบ่อย (`channel`) สัมพันธ์กับการตอบรับโปรโมชัน (`responded`) หรือไม่ จึงสร้างตารางไขว้ `pd.crosstab(cust['channel'], cust['responded'])` ของลูกค้าทุกคนในไฟล์ แล้วทดสอบความเป็นอิสระด้วย `stats.chi2_contingency` ค่าเริ่มต้น (ตาราง $2 \times 2$ จึงมี Yates continuity correction) ค่าสถิติไคสแควร์ $\chi^2$ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. สร้างตารางไขว้ 2x2 ระหว่าง channel กับ responded
cust = pd.read_csv('biz_customers.csv')
ct = pd.crosstab(cust['channel'], cust['responded'])
print("ตารางไขว้ (Contingency Table):")
print(ct)

# 2. ทดสอบความเป็นอิสระด้วย chi2_contingency (ค่าเริ่มต้นมี Yates correction)
res_77 = stats.chi2_contingency(ct)
print(f"Chi-square statistic: {res_77.statistic:.4f}")
print(f"p-value: {res_77.pvalue:.4f}")
print(f"คำตอบข้อ 7.7: {res_77.statistic:.4f}")


ตารางไขว้ (Contingency Table):
responded    0    1
channel            
online     503  255
store      329  116
Chi-square statistic: 7.1899
p-value: 0.0073
คำตอบข้อ 7.7: 7.1899


**โจทย์ข้อ 7.8:** ฝึกจากตารางไขว้ `pd.crosstab(cust['loyalty_member'], cust['churn'])` ของลูกค้าทุกคนในไฟล์ ภายใต้ $H_0$ (ไม่สัมพันธ์กัน) **จำนวนคาดหวัง (expected count)** ของช่องที่ `loyalty_member = 1` และ `churn = 1` เท่ากับ ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. สร้างตารางไขว้ loyalty_member กับ churn
cust = pd.read_csv('biz_customers.csv')
ct = pd.crosstab(cust['loyalty_member'], cust['churn'])
print("ตารางความถี่จริง (Observed):")
print(ct)

# 2. คำนวณความถี่คาดหวังด้วย chi2_contingency
res_78 = stats.chi2_contingency(ct)
expected = res_78.expected_freq
print("ตารางความถี่คาดหวัง (Expected):")
print(pd.DataFrame(expected, index=ct.index, columns=ct.columns))

# 3. ดึงค่าช่อง loyalty_member = 1, churn = 1
ans_78 = expected[1, 1]
print(f"จำนวนคาดหวัง (loyalty=1, churn=1): {ans_78:.3f}")
print(f"คำตอบข้อ 7.8: {ans_78:.3f}")


ตารางความถี่จริง (Observed):
churn             0    1
loyalty_member          
0               631  119
1               429   24
ตารางความถี่คาดหวัง (Expected):
churn                    0          1
loyalty_member                       
0               660.84788  89.152120
1               399.15212  53.847880
จำนวนคาดหวัง (loyalty=1, churn=1): 53.848
คำตอบข้อ 7.8: 53.848


**โจทย์ข้อ 7.9:** กรณีศึกษาทีมวิเคราะห์ข้อมูลของบริษัทบัตรสมาชิกต้องการคัดลูกค้าที่มีรายได้ต่อเดือน (พันบาท) `income_k` สูงผิดปกติไปตรวจสอบ โดยใช้กฎ IQR: ค่าที่มากกว่า $Q_3 + 1.5 \times \text{IQR}$ ถือว่าสูงผิดปกติ ($Q_1, Q_3$ หาด้วย `.quantile(0.25)`, `.quantile(0.75)` ของ pandas ค่าเริ่มต้น และ $\text{IQR} = Q_3 - Q_1$) จากข้อมูลของลูกค้าทุกคนในตาราง cust **จำนวนลูกค้าที่สูงผิดปกติ** = ? (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd

# 1. คำนวณ Q1, Q3 และ IQR ของ income_k
cust = pd.read_csv('biz_customers.csv')
q1 = cust['income_k'].quantile(0.25)
q3 = cust['income_k'].quantile(0.75)
iqr = q3 - q1
upper_threshold = q3 + 1.5 * iqr

print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"เกณฑ์ค่าผิดปกติระดับบน (Upper Threshold) = {upper_threshold:.2f}")

# 2. นับจำนวนลูกค้าที่มี income_k > upper_threshold
outliers = cust[cust['income_k'] > upper_threshold]
ans_79 = len(outliers)
print(f"จำนวนลูกค้าที่มีรายได้สูงผิดปกติ: {ans_79} คน")
print(f"คำตอบข้อ 7.9: {ans_79}")


Q1 = 19.95, Q3 = 37.35, IQR = 17.40
เกณฑ์ค่าผิดปกติระดับบน (Upper Threshold) = 63.45
จำนวนลูกค้าที่มีรายได้สูงผิดปกติ: 39 คน
คำตอบข้อ 7.9: 39


**โจทย์ข้อ 7.10:** ฝึกแบ่งลูกค้าเป็น 3 กลุ่มตาม `age`: `age < 40`, `40 <= age < 55`, `age >= 55` แล้วทดสอบความสัมพันธ์กับ `churn` ด้วย `stats.chi2_contingency` บนตารางไขว้ $3 \times 2$ (ลูกค้าทุกคนในไฟล์ `biz_customers.csv`, ค่าเริ่มต้น) ค่าสถิติไคสแควร์ $\chi^2$ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. แบ่งกลุ่มอายุเป็น 3 กลุ่มตามเงื่อนไข
cust = pd.read_csv('biz_customers.csv')
age_groups = pd.cut(
    cust['age'],
    bins=[-np.inf, 40, 55, np.inf],
    right=False,
    labels=['<40', '40-54', '>=55']
)
print("จำนวนลูกค้าในแต่ละกลุ่มอายุ:")
print(age_groups.value_counts())

# 2. สร้างตารางไขว้ 3x2 กับ churn
ct_3x2 = pd.crosstab(age_groups, cust['churn'])
print("\nตารางไขว้ 3x2:")
print(ct_3x2)

# 3. ทดสอบ chi-square test
res_710 = stats.chi2_contingency(ct_3x2)
print(f"\nChi-square statistic: {res_710.statistic:.4f}")
print(f"p-value: {res_710.pvalue:.4f}")
print(f"คำตอบข้อ 7.10: {res_710.statistic:.4f}")


จำนวนลูกค้าในแต่ละกลุ่มอายุ:
age
<40      638
40-54    481
>=55      84
Name: count, dtype: int64

ตารางไขว้ 3x2:
churn    0   1
age           
<40    581  57
40-54  407  74
>=55    72  12

Chi-square statistic: 11.3905
p-value: 0.0034
คำตอบข้อ 7.10: 11.3905


## สรุปทฤษฎีและแนวคิดหลักประจำ Week 7

---

### 1. การสำรวจข้อมูลเชิงสถิติ (EDA: Exploratory Data Analysis)

- **Pearson Correlation Coefficient ($r$):**
  $$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$
  วัดระดับความสัมพันธ์เชิงเส้นระหว่างสองตัวแปรเชิงปริมาณ ($-1 \le r \le 1$)

- **Skewness (ความเบ้):**
  $$\text{Skewness} = \frac{n}{(n-1)(n-2)} \sum \left(\frac{x_i - \bar{x}}{s}\right)^3$$
  - $\text{Skewness} = 0$: ข้อมูลสมมาตร
  - $\text{Skewness} > 0$: เบ้ขวา (หางยาวทางขวา ค่าเฉลี่ยมากกว่ามัธยฐาน)
  - $\text{Skewness} < 0$: เบ้ซ้าย

- **กฎ IQR ในการตรวจหาค่าผิดปกติ (Outlier Detection):**
  $$\text{IQR} = Q_3 - Q_1$$
  - ค่าที่ต่ำกว่า $Q_1 - 1.5 \times \text{IQR}$ หรือสูงกว่า $Q_3 + 1.5 \times \text{IQR}$ จัดเป็น Outliers

---

### 2. การทดสอบสมมติฐานด้วย t-test

- **One-Sample t-test:**
  $$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}$$
  ทดสอบว่าค่าเฉลี่ยของประชากรเท่ากับค่าเป้าหมาย $\mu_0$ หรือไม่

- **Two-Sample Independent t-test:**
  $$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{s_1^2/n_1 + s_2^2/n_2}}$$
  ทดสอบความแตกต่างระหว่างค่าเฉลี่ยของสองกลุ่มที่เป็นอิสระต่อกัน
  - การทดสอบแบบด้านเดียว (`alternative='greater'`): ทดสอบ $H_1: \mu_1 > \mu_2$

---

### 3. Bootstrap Resampling และ Percentile Confidence Interval

Bootstrap เป็นเทคนิค Non-parametric ที่ใช้สุ่มตัวอย่างซ้ำแบบใส่คืน (Sampling with replacement) ขนาด $n$ จากข้อมูลดั้งเดิมจำนวน $B$ รอบ เพื่อสร้างการกระจายตัวเชิงประจักษ์ (Empirical Distribution) ของค่าสถิติที่สนใจ โดยไม่ต้องอาศัยสมมติฐานเรื่องการแจกแจงแบบปกติ

- **$(1 - \alpha) \times 100\%$ Percentile Bootstrap CI:**
  $$[\hat{\theta}^*_{(\alpha/2)}, \; \hat{\theta}^*_{(1 - \alpha/2)}]$$
  สำหรับช่วงความเชื่อมั่น 90% ($\alpha = 0.10$):
  - ขอบล่าง: เปอร์เซ็นไทล์ที่ 5 (`np.percentile(data, 5)`)
  - ขอบบน: เปอร์เซ็นไทล์ที่ 95 (`np.percentile(data, 95)`)

---

### 4. Chi-Square Test for Independence (การทดสอบความเป็นอิสระ)

ใช้ตรวจสอบว่าตัวแปรเชิงกลุ่ม (Categorical variables) สองตัวมีความสัมพันธ์กันหรือไม่ บนตารางไขว้ (Contingency Table) ขนาด $r \times c$:

- **ความถี่คาดหวัง (Expected Frequency):**
  $$E_{ij} = \frac{R_i \times C_j}{N}$$
  เมื่อ $R_i$ คือผลรวมแถวที่ $i$, $C_j$ คือผลรวมคอลัมน์ที่ $j$, และ $N$ คือจำนวนข้อมูลทั้งหมด

- **ค่าสถิติ Chi-Square ($\chi^2$):**
  $$\chi^2 = \sum_{i=1}^r \sum_{j=1}^c \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$
  โดยมีองศาความเป็นอิสระ $df = (r - 1)(c - 1)$
- สำหรับตาราง $2 \times 2$ ฟังก์ชัน `stats.chi2_contingency` จะใช้ **Yates' Continuity Correction** โดยอัตโนมัติ:
  $$\chi^2_{\text{Yates}} = \sum \frac{(|O_{ij} - E_{ij}| - 0.5)^2}{E_{ij}}$$